# UWCT Tritonia hyperparameter tuning, round 1 (D-040): E025-E038

**Plan (D-040, declared in docs/EXPERIMENTS.md before any run):** baseline = E010's settings on frozen dataset v7
(seed 0 = E010 itself, seeds 1 / 2 = E025 / E026), and four arms that each change **one** setting from E010,
with seeds 0 / 1 / 2:

| arm | change | runs |
|---|---|---|
| baseline | seed only | E025, E026 (+ E010) |
| A | epochs 100 -> 200, patience 30 -> 50 | E027-E029 |
| B | batch 16 -> 8 | E030-E032 |
| C | scale 0.5 -> 0.2 | E033-E035 |
| D | imgsz 640 -> 1056 | E036-E038 |

- **Selection metric: val point AP** against the MTrackJ crosses (D-033), per arm the **mean over 3 seeds** with the
  seed range. An arm wins only if its mean beats the baseline mean by more than the larger of the two seed ranges.
- Ultralytics' P / R / mAP printed during training are against the fixed boxes: **not accuracy** (D-033).
- **No test split anywhere in this notebook** (CLAUDE.md rule 7). Frames are 15 s apart: sparse-frame/timelapse
  analysis, never "video tracking".

**How to run (interactive session; keep the browser tab open, about 1-2 h):**
1. Settings: Accelerator **GPU T4 x2**, **Internet on**; Add-ons -> Secrets -> `UWCT_GITHUB_TOKEN` (ticked for this notebook).
2. **Add Input** -> `anusidhu7/uwct-tritonia-v7-inputs`.
3. Cell 1: set `COMMIT` to the pushed commit that contains this notebook and the E025-E038 configs.
4. Run cells 1-7, check that cell 6 prints **VERIFIED** and cell 7 shows 5 passing smoke runs, then cell 8 (the queue),
   cell 9 (comparison) and cell 10 (download links). Download both zips before closing the session:
   saved versions have kept no output so far.

In [ ]:
# 1. Settings and helpers -- set COMMIT; keep everything else.
import os, re, subprocess, sys, json, time

REPO = "SakshamPainuly/underwater-creature-tracking"
BRANCH = "anu/e006-results"   # branch that holds COMMIT
COMMIT = "0000000000000000000000000000000000000000"  # REQUIRED: full sha, pushed
DATASET_VERSION = "v7"
KAGGLE_DATASET, BOXES_NAME, BATCH_NAME = "uwct-tritonia-v7-inputs", "tritonia_boxes_fixed2", "tritonia_label_batch1"
TOKEN_SECRET_NAME = "UWCT_GITHUB_TOKEN"

WORK = "/tmp"                          # scratch, not saved
REPO_DIR = f"{WORK}/uwct"
LOCAL_BOXES = f"{WORK}/raw/{BOXES_NAME}"
LOCAL_FRAMES = f"{WORK}/tritonia_frames"
LOCAL_BATCH = f"{WORK}/tritonia_batch"
DATASET_DIR = f"{REPO_DIR}/data/yolo/tritonia_{DATASET_VERSION}"
W = "/kaggle/working"
RUNS = f"{W}/runs"                     # run folders (zipped for download in cell 10)
WEIGHTS = f"{WORK}/weights"            # pretrained checkpoints
LOGS = f"{W}/tuning_logs"
os.makedirs(RUNS, exist_ok=True); os.makedirs(WEIGHTS, exist_ok=True); os.makedirs(LOGS, exist_ok=True)

assert "KAGGLE_KERNEL_RUN_TYPE" in os.environ, "this notebook is for Kaggle (GPU T4 x2)"
assert re.fullmatch(r"[0-9a-f]{40}", COMMIT) and set(COMMIT) != {"0"}, "Set COMMIT to the exact 40-character commit sha to run"


def sh(*cmd, env=None):
    """Run a command in the repo, stream its output, raise if it fails."""
    p = subprocess.Popen(list(cmd), cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"FAILED (exit {p.returncode}): {' '.join(cmd)}")


def get_secret(name):
    from kaggle_secrets import UserSecretsClient
    return UserSecretsClient().get_secret(name)


print("settings OK | runs:", RUNS)

In [ ]:
# 2. GPUs and the attached v7 inputs (read-only under /kaggle/input; exactly one match each).
import shutil
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "NO GPU -- switch the accelerator to GPU T4 x2")


def find_input_dir(name, must_contain):
    hits = sorted(d for d, subdirs, _ in os.walk("/kaggle/input")
                  if os.path.basename(d) == name and must_contain in subdirs and KAGGLE_DATASET in d)
    assert len(hits) == 1, (f"expected one {name}/ folder (with {must_contain}/) of dataset {KAGGLE_DATASET} "
                            f"under /kaggle/input, found {hits} -- attach the dataset (Add Input)")
    return hits[0]


SRC_BOXES = find_input_dir(BOXES_NAME, "labels")
SRC_BATCH = find_input_dir(BATCH_NAME, "images")
print("label source:", SRC_BOXES, "| batch:", SRC_BATCH)

In [ ]:
# 3. Check out the exact commit (read-only token from Kaggle Secrets, as in colab_train_tritonia.ipynb cell 4:
#    the token is only in the environment of the one git process, never in a URL, file, .git/config or output).
import stat

REPO_URL = f"https://github.com/{REPO}.git"
ASKPASS = f"{WORK}/uwct_git_askpass.sh"
with open(ASKPASS, "w") as f:
    f.write('#!/bin/sh\n'
            'case "$1" in\n'
            '  Username*) echo "x-access-token" ;;\n'
            '  *) printf "%s\\n" "$UWCT_GIT_TOKEN" ;;\n'
            'esac\n')
os.chmod(ASKPASS, stat.S_IRWXU)


def git(*args, cwd=REPO_DIR):
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"git {args[0]} failed:\n{r.stderr}")
    return r.stdout.strip()


def git_remote(*args, cwd=REPO_DIR):
    env = {**os.environ, "GIT_ASKPASS": ASKPASS, "GIT_TERMINAL_PROMPT": "0", "UWCT_GIT_TOKEN": get_secret(TOKEN_SECRET_NAME)}
    try:
        r = subprocess.run(["git", "-c", "credential.helper=", *args], cwd=cwd, env=env, capture_output=True, text=True)
        err = r.stderr.replace(env["UWCT_GIT_TOKEN"], "***")
        config = os.path.join(REPO_DIR, ".git", "config")
        leaked = os.path.isfile(config) and env["UWCT_GIT_TOKEN"] in open(config).read()
    finally:
        env.clear()
    if r.returncode != 0:
        raise RuntimeError(f"git {args[0]} failed:\n{err}")
    assert not leaked, "token found in .git/config"


if not os.path.isdir(REPO_DIR):
    git_remote("clone", "--branch", BRANCH, REPO_URL, REPO_DIR, cwd=WORK)
else:
    git_remote("fetch", "origin", BRANCH)
git("checkout", "--detach", COMMIT)
assert git("rev-parse", "HEAD") == COMMIT
print("commit:", COMMIT, "| working tree:", git("status", "--porcelain") or "clean")
open(f"{LOGS}/code_version.txt", "w").write(f"{REPO_URL}\n{BRANCH}\n{COMMIT}\n")

In [ ]:
# 4. Pinned GPU requirements (ultralytics==8.4.155; torch/CUDA from Kaggle) and the session environment report.
sh(sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-gpu.txt")
sh(sys.executable, "-c", "import ultralytics, torch; print('ultralytics', ultralytics.__version__, '| torch', torch.__version__)")
sh(sys.executable, "scripts/env_report.py", "--out", f"{LOGS}/env_report_session.json")

In [ ]:
# 5. Local copies of the label source, frames and batch (nothing is written to /kaggle/input); the label source is a
#    read-only raw root (UWCT_TRITONIA_BOXES_ROOT) for every later script. Then the frozen-registry check of v7.
for src, dst, ignore in ((SRC_BOXES, LOCAL_BOXES, None), (os.path.join(SRC_BATCH, "images"), LOCAL_FRAMES, None),
                         (SRC_BATCH, LOCAL_BATCH, shutil.ignore_patterns("hints"))):
    if os.path.isdir(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst, ignore=ignore)
os.environ["UWCT_TRITONIA_BOXES_ROOT"] = LOCAL_BOXES
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from dataset_audit.versioning import read_version_meta, verify_frozen
verify_frozen(DATASET_VERSION)
meta = read_version_meta(DATASET_VERSION)
assert meta["dataset_kind"] == "tritonia" and meta["status"] == "frozen", meta
print(len(os.listdir(os.path.join(LOCAL_BOXES, "labels"))), "label files |",
      len([f for f in os.listdir(LOCAL_FRAMES) if f.endswith(".png")]), "frames | v7 frozen OK")

In [ ]:
# 6. Build the YOLO view of v7 (every file hash-checked), check its fingerprint, Ultralytics check (must print VERIFIED),
#    and check all 14 D-040 configs at this commit (dataset v7, this fingerprint, parent E010).
sh(sys.executable, "scripts/prepare_tritonia_yolo.py", "--version", DATASET_VERSION,
   "--frames", LOCAL_FRAMES, "--boxes", LOCAL_BOXES)
FP = json.load(open(os.path.join(DATASET_DIR, "build_summary.json")))["dataset_fingerprint"]
assert FP == "8ea6503cab38c9fed92f1c5be9d71fae5de6af3d43f17a31d533d8bf7854649a", FP
sh(sys.executable, "scripts/verify_yolo_dataset.py", "--dataset", DATASET_DIR)

from training.experiment import load_config, validate_config
ARMS = {"baseline": ["E025", "E026"], "A": ["E027", "E028", "E029"], "B": ["E030", "E031", "E032"],
        "C": ["E033", "E034", "E035"], "D": ["E036", "E037", "E038"]}
CFG_DIR = os.path.join(REPO_DIR, "configs", "experiments")
CONFIGS = {}
for arm, ids in ARMS.items():
    for e in ids:
        (name,) = [f for f in os.listdir(CFG_DIR) if f.startswith(e + "_")]
        cfg = validate_config(load_config(os.path.join(CFG_DIR, name)), name)
        assert cfg["dataset_version"] == DATASET_VERSION and cfg["dataset_fingerprint"] == FP and cfg["parent"] == "E010", cfg
        CONFIGS[e] = dict(arm=arm, name=name, seed=cfg["seed"], imgsz=cfg["train_args"]["imgsz"], args=cfg["train_args"])
for e, c in CONFIGS.items():
    print(e, c["arm"], "seed", c["seed"], c["args"])

In [ ]:
# 7. Smoke tests (CLAUDE.md section 6): 1 epoch on 10% of train, validated on val, for one config of each arm,
#    two at a time (one per T4). Every training-side check runs (config, clean commit, frozen v7 view, checkpoint).
#    NOT results. Smoke folders are RUNS/<id>_smoke_<utc>/.
import threading
RUN_SMOKE = False  # set to True deliberately
assert RUN_SMOKE, "Smoke test not enabled"
N_GPUS = len([l for l in subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.splitlines()
              if l.startswith("GPU ")])
LANES = max(1, min(N_GPUS, 2))
ENV = dict(os.environ)  # includes UWCT_TRITONIA_BOXES_ROOT (cell 5)


def train_cmd(e, *extra):
    return [sys.executable, f"{REPO_DIR}/scripts/train_yolo.py", "--config", f"configs/experiments/{CONFIGS[e]['name']}",
            "--runs-root", RUNS, "--weights-dir", WEIGHTS, "--dataset", DATASET_DIR, *extra]


def run_on_gpu(cmd, gpu, log_path):
    with open(log_path, "w") as log:
        return subprocess.run(cmd, cwd=REPO_DIR, stdout=log, stderr=subprocess.STDOUT, text=True,
                              env=dict(ENV, CUDA_VISIBLE_DEVICES=str(gpu))).returncode


smoke_ids = ["E025", "E027", "E030", "E033", "E036"]
smoke = {}
def smoke_lane(gpu, ids):
    for e in ids:
        t0 = time.time()
        rc = run_on_gpu(train_cmd(e, "--smoke"), gpu, f"{LOGS}/smoke_{e}.log")
        smoke[e] = dict(exit=rc, seconds=round(time.time() - t0))
        print(f"smoke {e} (arm {CONFIGS[e]['arm']}) GPU {gpu}: exit {rc}, {smoke[e]['seconds']} s", flush=True)
threads = [threading.Thread(target=smoke_lane, args=(g, smoke_ids[g::LANES])) for g in range(LANES)]
[t.start() for t in threads]; [t.join() for t in threads]
json.dump(smoke, open(f"{LOGS}/smoke_summary.json", "w"), indent=1)
bad = [e for e, s in smoke.items() if s["exit"] != 0]
for e in bad:
    print(f"--- {e} log tail:\n" + "\n".join(open(f"{LOGS}/smoke_{e}.log", errors="replace").read().splitlines()[-15:]))
assert not bad, f"smoke failed: {bad}"
print("ALL 5 SMOKE RUNS PASSED")

In [ ]:
# 8. D-040 round 1: train E025-E038 on both GPUs (one process per T4), seed 0 of every arm first, longest (D) first.
#    After each run: val point eval of its best.pt (all 27 val frames, threshold = its best val F1, at its own imgsz).
#    Val only; the test split is never touched. Status every 10 min; RUNS/queue_status.json is updated as it goes.
RUN_QUEUE = False  # set to True deliberately, after cell 7 passed
assert RUN_QUEUE, "Queue not enabled"
QUEUE = ["E036", "E027", "E030", "E033", "E025",
         "E037", "E028", "E031", "E034", "E026",
         "E038", "E029", "E032", "E035"]
assert sorted(QUEUE) == sorted(CONFIGS), "queue and configs differ"
pending = list(QUEUE)
status = {e: {"arm": CONFIGS[e]["arm"], "seed": CONFIGS[e]["seed"], "state": "queued"} for e in QUEUE}
LOCK = threading.Lock()
T_QUEUE = time.time()


def save_status():
    json.dump({"elapsed_min": round((time.time() - T_QUEUE) / 60, 1), "runs": status},
              open(f"{LOGS}/queue_status.json", "w"), indent=1)


def lane(gpu):
    while True:
        with LOCK:
            if not pending:
                return
            e = pending.pop(0)
            status[e].update(state="training", gpu=gpu)
            save_status()
        print(f"START {e} (arm {CONFIGS[e]['arm']}, seed {CONFIGS[e]['seed']}) on GPU {gpu}", flush=True)
        t0 = time.time()
        rc = run_on_gpu(train_cmd(e), gpu, f"{LOGS}/train_{e}.log")
        rec_path = f"{RUNS}/{e}/run_record.json"
        if rc != 0 or not os.path.exists(rec_path):
            tail = open(f"{LOGS}/train_{e}.log", errors="replace").read().splitlines()[-15:]
            with LOCK:
                status[e].update(state=f"failed training (exit {rc})", minutes=round((time.time() - t0) / 60, 1), log_tail=tail)
                save_status()
            print(f"FAILED {e} exit {rc}:\n  " + "\n  ".join(tail), flush=True)
            continue
        rec = json.load(open(rec_path))
        cmd = [sys.executable, f"{REPO_DIR}/scripts/tritonia_point_eval.py", "--batch", LOCAL_BATCH, "--split", "val",
               "--weights", rec["outputs"]["best"]["path"], "--imgsz", str(CONFIGS[e]["imgsz"]),
               "--pred-dir", f"{RUNS}/{e}/val_point_predictions", "--out", f"{RUNS}/{e}/val_point_scores.json"]
        rc2 = run_on_gpu(cmd, gpu, f"{LOGS}/pointeval_{e}.log")
        with LOCK:
            if rc2 == 0:
                v = json.load(open(f"{RUNS}/{e}/val_point_scores.json"))
                status[e].update(state="finished", minutes=round((time.time() - t0) / 60, 1),
                                 epochs=rec.get("epochs_completed"), point_ap=v["point_ap"], at_conf=v["at_conf"],
                                 ultralytics_val=rec["val_metrics"], best_sha256=rec["outputs"]["best"]["sha256"])
            else:
                status[e].update(state=f"trained; point eval failed (exit {rc2})")
            save_status()
        if rc2 == 0:
            a = v["at_conf"]
            print(f"DONE  {e} arm {CONFIGS[e]['arm']} seed {CONFIGS[e]['seed']}: {status[e]['minutes']} min, "
                  f"{rec.get('epochs_completed')} epochs | VAL point AP {v['point_ap']:.3f} | P/R/F1 {a['precision']:.3f}/"
                  f"{a['recall']:.3f}/{a['f1']:.3f} at conf {a['conf']}", flush=True)
        else:
            print(f"POINT EVAL FAILED {e}: see {LOGS}/pointeval_{e}.log", flush=True)


print(f"{LANES} lane(s); {len(QUEUE)} runs; {time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime())}")
threads = [threading.Thread(target=lane, args=(g,), daemon=True) for g in range(LANES)]
[t.start() for t in threads]
while any(t.is_alive() for t in threads):
    for t in threads:
        t.join(timeout=300)
    with LOCK:
        busy = [e for e, s in status.items() if s["state"] == "training"]
        done = sum(s["state"] == "finished" for s in status.values())
    print(f"  .. {round((time.time() - T_QUEUE) / 60)} min: {done}/{len(QUEUE)} finished; training {busy}", flush=True)
save_status()
print("\nQUEUE FINISHED")
for e, s in status.items():
    print(f"  {e} arm {s['arm']} seed {s['seed']}: {s['state']}" + (f"  point AP {s['point_ap']:.3f}" if "point_ap" in s else ""))

In [ ]:
# 9. Round-1 comparison (val only; D-040 point 4): per arm the mean val point AP over its 3 seeds, the seed range, and
#    the declared rule. Baseline seed 0 = E010 (registered; its val point AP 0.496 at conf 0.16, docs/EXPERIMENTS.md).
#    Also writes RUNS/d040_round1_summary.json and a bar chart PNG. Claude recomputes this from the files before any decision.
import statistics
st = json.load(open(f"{LOGS}/queue_status.json"))["runs"]
E010 = {"point_ap": 0.496, "at_conf": {"conf": 0.16, "precision": 0.614, "recall": 0.4575, "f1": 0.5243}}
arms = {}
for arm, ids in ARMS.items():
    vals = [(e, st[e]) for e in ids if st.get(e, {}).get("state") == "finished"]
    if arm == "baseline":
        vals = [("E010", E010)] + vals
    arms[arm] = vals
summary = {}
base = [v["point_ap"] for _, v in arms["baseline"]]
b_mean, b_range = statistics.mean(base), max(base) - min(base)
rows = ["| arm | runs | val point AP mean | seed range | per seed | mean F1 | wins? |", "|---|---:|---:|---:|---|---:|---|"]
for arm, vals in arms.items():
    aps = [v["point_ap"] for _, v in vals]
    if not aps:
        rows.append(f"| {arm} | 0 | - | - | - | - | no runs |"); continue
    mean, rng = statistics.mean(aps), max(aps) - min(aps)
    f1 = statistics.mean(v["at_conf"]["f1"] for _, v in vals)
    if arm == "baseline":
        verdict = "reference"
    elif len(aps) < 3:
        verdict = "cannot win (fewer than 3 seeds)"
    else:
        verdict = "WINS" if mean - b_mean > max(rng, b_range) else "no evidence of a gain"
    summary[arm] = dict(runs=[e for e, _ in vals], point_ap=aps, mean=round(mean, 4), range=round(rng, 4),
                        mean_f1=round(f1, 4), verdict=verdict)
    rows.append(f"| {arm} | {len(aps)} | {mean:.3f} | {rng:.3f} | {', '.join(f'{e} {a:.3f}' for (e, _), a in zip(vals, aps))} "
                f"| {f1:.3f} | {verdict} |")
from IPython.display import Markdown, display
display(Markdown("## D-040 round 1 -- val point AP (not mAP; vs the MTrackJ crosses)\n" + "\n".join(rows) +
                 f"\n\nRule: an arm wins if its mean exceeds the baseline mean ({b_mean:.3f}) by more than the larger of "
                 f"its own and the baseline's seed range (baseline range {b_range:.3f})."))
json.dump({"baseline_mean": b_mean, "baseline_range": b_range, "arms": summary}, open(f"{RUNS}/d040_round1_summary.json", "w"), indent=1)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
names = [a for a in arms if arms[a]]
fig, ax = plt.subplots(figsize=(7, 4))
for i, a in enumerate(names):
    aps = [v["point_ap"] for _, v in arms[a]]
    ax.bar(i, statistics.mean(aps), color="#888" if a == "baseline" else "#3a7bd5", alpha=0.8)
    ax.scatter([i] * len(aps), aps, color="k", s=14, zorder=3)
ax.set_xticks(range(len(names)), [{"baseline": "baseline\n(E010 settings)", "A": "A\nepochs 200", "B": "B\nbatch 8",
                                     "C": "C\nscale 0.2", "D": "D\nimgsz 1056"}[a] for a in names])
ax.set_ylabel("val point AP (mean of seeds; dots = seeds)")
ax.set_title("Tritonia tuning round 1 (D-040), 27 val frames")
fig.tight_layout(); fig.savefig(f"{RUNS}/d040_round1_val_point_ap.png", dpi=150)
display(fig)

In [ ]:
# 10. Download links (saved versions have kept no output so far, so download now):
#     d040_results.zip  = run records, val point scores, results.csv, args.yaml, logs, summary + chart (small)
#     d040_weights.zip  = best.pt of every finished run (about 20-40 MB each)
import zipfile
from IPython.display import FileLink
os.chdir(W)
with zipfile.ZipFile("d040_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for d, _, files in os.walk("runs"):
        for n in sorted(files):
            p = os.path.join(d, n)
            if n.endswith(".pt") or n.endswith(".jpg") or "_smoke_" in p:
                continue
            z.write(p)
    for d, _, files in os.walk("tuning_logs"):
        for n in sorted(files):
            z.write(os.path.join(d, n))
with zipfile.ZipFile("d040_weights.zip", "w", zipfile.ZIP_STORED) as z:
    for e in sorted(CONFIGS):
        p = f"runs/{e}/train/weights/best.pt"
        if os.path.exists(p):
            z.write(p)
for f in ("d040_results.zip", "d040_weights.zip"):
    print(f, os.path.getsize(f), "bytes")
    display(FileLink(f))